# Training & Evaluation - Classifying Spam Emails

Notebook này tổng hợp phần train model spam/not spam cho project: kiểm tra dữ liệu clean, thống kê nhãn, train 3 model cơ bản và xem metric sau train.

Ghi chú: notebook đang dùng `SAMPLE_SIZE = 60000` để chạy nhanh trên máy cá nhân. Đổi thành `None` nếu muốn train toàn bộ `data/processed/combined_balanced_clean.csv`.

## 1. Kết quả lọc dữ liệu hiện tại

- Input rows: `296196`
- Clean rows: `294718`
- Removed rows: `1478`
- Label trước lọc: `{'0': 148098, '1': 148098}`
- Label sau lọc/cân bằng: `{'0': 147359, '1': 147359}`
- Lỗi chính: `{'short_text': 1156}`

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "combined_balanced_clean.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

if not DATA_PATH.exists():
    from src.data_quality import clean_balanced_dataset

    print("Không thấy file clean, đang tạo lại bằng src.data_quality...")
    clean_balanced_dataset()

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Data exists:", DATA_PATH.exists())

In [ ]:
dataset = pd.read_csv(DATA_PATH)
print(dataset.shape)
dataset.head()

In [ ]:
label_counts = dataset["label"].value_counts().sort_index().rename(index={0: "not spam", 1: "spam"})
label_percent = (label_counts / label_counts.sum() * 100).round(2)
pd.DataFrame({"count": label_counts, "percent": label_percent})

## 2. Kiểm tra naming/source code

- Số file Python đã kiểm tra: `11`
- Số lỗi đặt tên: `0`
- Quy tắc: file/function/variable dùng `snake_case`, constant dùng `UPPER_CASE`, class dùng `PascalCase`.

In [ ]:
style_report_path = REPORTS_DIR / "code_style_name_check.json"
style_report = json.loads(style_report_path.read_text(encoding="utf-8"))
style_report["naming_issues"]

## 3. Train model

Ba model được train cùng pipeline `TfidfVectorizer + classifier`:

- `naive_bayes`: nhanh, baseline tốt cho text classification.
- `logistic_regression`: cân bằng giữa tốc độ, độ chính xác và khả năng giải thích.
- `linear_svm`: thường mạnh với TF-IDF và dữ liệu văn bản lớn.

Metric ưu tiên là `recall_spam` và `f1_spam` vì bỏ sót spam nguy hiểm hơn chặn nhầm một số email tốt.

In [ ]:
import sys
sys.path.append(str(PROJECT_ROOT))

from src.model_train import run_modeling_pipeline

SAMPLE_SIZE = 60000  # Đổi thành None nếu muốn train toàn bộ dataset clean.
metrics_table = run_modeling_pipeline(sample_size=SAMPLE_SIZE)
metrics_table

## 4. Metric sau train gần nhất

| model               | accuracy | precision_spam | recall_spam | f1_spam  |
| ------------------- | -------- | -------------- | ----------- | -------- |
| linear_svm          | 0.979167 | 0.979007       | 0.979333    | 0.97917  |
| logistic_regression | 0.974    | 0.970394       | 0.977833    | 0.974099 |
| naive_bayes         | 0.963417 | 0.977339       | 0.948833    | 0.962875 |

In [ ]:
metrics_path = REPORTS_DIR / "model_metrics.csv"
metrics = pd.read_csv(metrics_path)
metrics.sort_values(["f1_spam", "recall_spam"], ascending=False)

In [ ]:
from src.model_evaluate import evaluate_from_predictions_csv

comparison = evaluate_from_predictions_csv(
    predictions_csv=REPORTS_DIR / "model_predictions.csv",
    true_col="label",
    pred_cols=["naive_bayes", "logistic_regression", "linear_svm"],
    output_dir=FIGURES_DIR,
)
comparison

## 5. Confusion matrix

Các ảnh dưới đây được sinh tự động từ `src/model_evaluate.py`. Nếu nhóm muốn dùng ảnh ngoài cho báo cáo cuối, có thể thay các file trong `reports/figures/`.

In [ ]:
from IPython.display import Image, display

for name in ["naive_bayes", "logistic_regression", "linear_svm"]:
    image_path = FIGURES_DIR / f"{name}_confusion_matrix.png"
    print(name, image_path)
    if image_path.exists():
        display(Image(filename=str(image_path)))

## 6. Demo predict

File `src/predict.py` load model tốt nhất từ `models/spam_classifier.joblib` và trả về nhãn `spam` hoặc `not spam`, kèm `spam_score` trong khoảng 0-1.

In [ ]:
from src.predict import predict_email

sample_emails = [
    "Congratulations! You won a free iPhone. Click now to claim your prize.",
    "Hi team, please confirm your availability for the meeting tomorrow.",
    "Urgent account warning: verify your password immediately or your mailbox will be suspended.",
]

[predict_email(email) for email in sample_emails]

## 7. Kết luận nhanh

- Dataset clean hiện cân bằng 2 class và đã loại dòng quá ngắn/lỗi.
- `linear_svm` đang là model tốt nhất trong lần train mẫu 60,000 dòng.
- Khi chốt báo cáo, nhóm có thể train lại với `SAMPLE_SIZE = None` để dùng toàn bộ dataset.